# 02 — Model Training (LightGBM Multimodal Ensemble)
**Alzheimer's Disease Classification | KLE Technological University**

Trains two separate LightGBM classifiers (cognitive + MRI) and combines them via weighted probability ensemble.

### Input
- `ADNI_MRI_Cognitive_Merged.csv` — generated by `01_data_preprocessing.ipynb`

### Output
- `lgb_model_cog.joblib` — trained cognitive LightGBM model
- `lgb_model_mri.joblib` — trained MRI LightGBM model
- `scaler_cog.joblib` / `scaler_mri.joblib` — fitted scalers
- `ensemble_params.joblib` — ensemble weights + decision threshold

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, precision_score, recall_score, f1_score
)
from sklearn.utils.class_weight import compute_sample_weight
from lightgbm import LGBMClassifier
import joblib
import warnings
warnings.filterwarnings('ignore')

# ---- Paths ----
MERGED_PATH = '/content/drive/MyDrive/AD1/ADNI_MRI_Cognitive_Merged.csv'
OUTPUT_DIR  = '/content/drive/MyDrive/AD1/lightgbm_final'
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load and clean
df = pd.read_csv(MERGED_PATH, low_memory=False)
print('Loaded:', df.shape)

if 'APOE4' in df.columns:
    df = df.drop(columns=['APOE4'])

# Label mapping: CN/SMC -> 0, MCI variants -> 1, AD -> 2
label_map = {'CN':0, 'MCI':1, 'LMCI':1, 'EMCI':1, 'SMC':0, 'AD':2}
df['DX_num'] = df['DX_bl'].map(label_map)
df = df.dropna(subset=['DX_num'])
df['DX_num'] = df['DX_num'].astype(int)
print('Class distribution:')
print(df['DX_num'].value_counts().rename({0:'CN', 1:'MCI', 2:'AD'}))

In [ ]:
# Feature engineering
required = ['AGE','PTEDUCAT','MMSE','ADAS13','CDRSB','Hippocampus','Entorhinal']
for r in required:
    if r not in df.columns:
        raise ValueError(f'Missing column: {r}')

df['ADAS_gap']  = df['MMSE'] - df['ADAS13']
df['AGE_sq']    = df['AGE'] ** 2
df['H_E_ratio'] = df['Hippocampus'] / (df['Entorhinal'] + 1e-9)

cog_features = ['AGE','AGE_sq','PTEDUCAT','MMSE','ADAS13','CDRSB','ADAS_gap']
mri_features = ['Hippocampus','Entorhinal','H_E_ratio']

mask = df[cog_features + mri_features].notna().all(axis=1)
df   = df[mask].reset_index(drop=True)
print('After cleaning:', df.shape)

In [ ]:
# Stratified 80/20 train-test split
Xc = df[cog_features].copy()
Xm = df[mri_features].copy()
y  = df['DX_num'].copy()

Xc_train, Xc_test, Xm_train, Xm_test, y_train, y_test = train_test_split(
    Xc, Xm, y, test_size=0.20, random_state=42, stratify=y
)
print(f'Train: {len(y_train)} | Test: {len(y_test)}')

In [ ]:
# Z-score normalization (fit on train only)
scaler_cog = StandardScaler().fit(Xc_train)
scaler_mri = StandardScaler().fit(Xm_train)

Xc_train_s = pd.DataFrame(scaler_cog.transform(Xc_train), columns=cog_features, index=Xc_train.index)
Xc_test_s  = pd.DataFrame(scaler_cog.transform(Xc_test),  columns=cog_features, index=Xc_test.index)
Xm_train_s = pd.DataFrame(scaler_mri.transform(Xm_train), columns=mri_features, index=Xm_train.index)
Xm_test_s  = pd.DataFrame(scaler_mri.transform(Xm_test),  columns=mri_features, index=Xm_test.index)

In [ ]:
# Balanced sample weights to handle class imbalance
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

# LightGBM hyperparameters
params = {
    'n_estimators' : 400,
    'learning_rate': 0.03,
    'max_depth'    : 6,
    'random_state' : 42,
    'num_class'    : 3,
    'objective'    : 'multiclass'
}

print('Training LightGBM — Cognitive model...')
model_cog = LGBMClassifier(**params)
model_cog.fit(Xc_train_s, y_train, sample_weight=sample_weights)

print('Training LightGBM — MRI model...')
model_mri = LGBMClassifier(**params)
model_mri.fit(Xm_train_s, y_train, sample_weight=sample_weights)
print('Training complete.')

In [ ]:
# Weighted ensemble: P_ens = 0.6 * P_cog + 0.4 * P_mri
w_cog, w_mri = 0.6, 0.4

probs_cog = model_cog.predict_proba(Xc_test_s)
probs_mri = model_mri.predict_proba(Xm_test_s)
probs_ens = w_cog * probs_cog + w_mri * probs_mri
y_pred    = np.argmax(probs_ens, axis=1)

print('\n--- Ensemble Evaluation (threshold = 0.50) ---')
print('Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('AUC (OvR):', round(roc_auc_score(y_test, probs_ens, multi_class='ovr'), 4))
print('\nClassification Report:')
print(classification_report(y_test, y_pred, target_names=['CN','MCI','AD']))

In [ ]:
# AD-specific threshold tuning
y_test_ad = (y_test == 2).astype(int)
ad_prob   = probs_ens[:, 2]

precisions, recalls, thresholds = precision_recall_curve(y_test_ad, ad_prob)

min_precision = 0.60
candidates = [(p,r,t) for p,r,t in zip(precisions[:-1], recalls[:-1], thresholds) if p >= min_precision]

if candidates:
    best = max(candidates, key=lambda x: x[1])
    chosen_threshold = best[2]
else:
    f1s = 2*precisions*recalls/(precisions+recalls+1e-12)
    chosen_threshold = thresholds[np.nanargmax(f1s[:-1])]

print(f'Chosen AD threshold: {round(chosen_threshold, 4)}')

In [ ]:
# Save all artifacts
joblib.dump(model_cog,  os.path.join(OUTPUT_DIR, 'lgb_model_cog.joblib'))
joblib.dump(model_mri,  os.path.join(OUTPUT_DIR, 'lgb_model_mri.joblib'))
joblib.dump(scaler_cog, os.path.join(OUTPUT_DIR, 'scaler_cog.joblib'))
joblib.dump(scaler_mri, os.path.join(OUTPUT_DIR, 'scaler_mri.joblib'))
joblib.dump(
    {'w_cog': w_cog, 'w_mri': w_mri, 'chosen_threshold': chosen_threshold},
    os.path.join(OUTPUT_DIR, 'ensemble_params.joblib')
)
print('All artifacts saved to:', OUTPUT_DIR)